# SOLA on pixels: what changes when the pixels shrink?

This is the **same experiment** as the existing discretisation demo: the same
600 rays, smooth truth and one noisy data vector, copied unchanged in `problem.py`.
We now estimate one property with SOLA rather than reconstruct a model.

The requested property is fixed: the mean over the disk centred at $(0.3,0.65)$,
radius $0.12$. The main calculation constrains the delivered functional to return
one on a constant model. An unconstrained calculation is included for comparison.
We do not tune weights on the realised observations.

The question is whether refining a pixel representation preserves the selected
question and its interpretation. No outcome is assumed in advance.

In [ ]:
%matplotlib inline
import json, time, socket, hashlib, subprocess, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import sola_demo as S
P = S.P
OUT = Path('output'); OUT.mkdir(exist_ok=True)
src, sta, p, q = P.geometry()
d_clean, d = P.data()
truth = S.target_truth()
print('machine:', socket.gethostname())
print('Python:', sys.executable)
print('code commit:', subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True).stdout.strip())
print('problem.py SHA256:', hashlib.sha256(Path('problem.py').read_bytes()).hexdigest())
print(f'{len(d)} original rays, noise sigma={P.SIGMA}; fixed disk mean={truth:.12f}')
print(f'disk centre={S.CENTRE}, radius={S.RADIUS}; fixed dimensionless lambda={S.LAMBDA}')


In [ ]:
z=np.linspace(0,1,200); X,Y=np.meshgrid(z,z)
fig,axs=plt.subplots(1,2,figsize=(10,4))
for a,b in zip(p,q): axs[0].plot([a[0],b[0]],[a[1],b[1]],color='.4',lw=.3,alpha=.2)
axs[0].scatter(*src.T,s=15,marker='*'); axs[0].scatter(*sta.T,s=15,marker='^')
axs[0].set_title('The original 600 rays')
im=axs[1].imshow(P.true_model(X,Y),origin='lower',extent=[0,1,0,1],cmap='RdBu_r')
axs[1].add_patch(plt.Circle(S.CENTRE,S.RADIUS,fill=False,color='k'))
axs[1].set_title('Known truth and the fixed target disk')
for ax in axs: ax.set_aspect('equal'); ax.set_xlim(0,1); ax.set_ylim(0,1)
fig.colorbar(im,ax=axs[1]); plt.show()

## The pixel SOLA problem

Each entry of $G_h$ is the exact length of a ray in a pixel. Pixel area is $a=h^2$.
The sensitivity density is $G_{ij}/a$, and the delivered density is
$r_h=G_h^T x/a$. The target $t_h$ uses disk/pixel overlap areas, not a
centre-in-disk mask. Its integral is one on every mesh.

The data Gram and target/data pairings are
$$K_h=G_hG_h^T/a,\qquad b_h=G_ht_h.$$
We minimize
$$\frac{\|t_h-r_h\|_{L^2}^2}{\|t\|_{L^2}^2}
 +\lambda\frac{\sigma^2\|x\|^2}{\sigma_{\rm ref}^2},
 \qquad v^Tx=1,$$
where $v_i$ is the full ray length, $\sigma_{\rm ref}=\sigma=0.01$,
$\|t\|_{L^2}^2=1/(\pi R^2)$ and $\lambda=0.1$ are held fixed.
Using the full target norm rather than its projected norm only adds a constant
to the fixed-$\lambda$ objective. The solve is
$$H_h=K_h+\lambda\|t\|^2 I,\quad x_0=H_h^{-1}b_h,\quad
x=x_0+H_h^{-1}v\frac{1-v^Tx_0}{v^TH_h^{-1}v}.$$
`sola_demo.py` implements this correction; tests compare it against an independent
KKT system. There is no clipping of eigenvalues or data-misfit tuning.

In [ ]:
start=time.perf_counter()
rows=S.pixel_study()
S.save_json(OUT/'pixel_results.json',dict(lambda_value=S.LAMBDA,rows=S.json_rows(rows)))
np.savez_compressed(OUT/'pixel_weights.npz',**{f'n{r["n"]}':r['x'] for r in rows})
print(f'Completed in {time.perf_counter()-start:.1f}s')
print('\n n     mass      target mass   estimate   delivered truth   sigma      target bias   ||r_h||L2   relative mismatch')
for r in rows:
 print(f"{r['n']:4d} {r['mass']:10.7f} {r['target_mass']:10.7f} {r['estimate']:10.5f} {r['delivered_truth']:12.5f} "
       f"{r['std']:10.4g} {r['target_bias']:11.5f} {r['kernel_norm']:11.4g} {r['relative_mismatch']:10.4g}")

## Spatial interpretation on each mesh

These are **Lebesgue density kernels for the pixel model**. The target is a disk
density; the delivered kernel is built from the pixels touched by weighted rays.
The display is clipped to one common colour range so that changing peak amplitudes
does not hide the spatial support. The printed norms and peak labels remain unclipped.

In [ ]:
chosen=[rows[i] for i in (0,2,4,len(rows)-1)]
fig,axs=plt.subplots(1,len(chosen),figsize=(15,3.8),constrained_layout=True)
lim=2/S.AREA
for ax,r in zip(axs,chosen):
 im=ax.imshow(r['density'],origin='lower',extent=[0,1,0,1],cmap='RdBu_r',vmin=-lim,vmax=lim)
 ax.add_patch(plt.Circle(S.CENTRE,S.RADIUS,fill=False,color='k',lw=1))
 ax.set_title(f"{r['n']} x {r['n']}\nmax |density|={np.abs(r['density']).max():.1f}")
 ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im,ax=axs,shrink=.8,extend='both',label='pixel density (colour-clipped)'); plt.show()

## Weights, uncertainty and kernel mismatch

Each output is a noisy observation of its own delivered property:
$$x_h^Td=x_h^TGm+x_h^T\eta.$$
The physical delivered truth below is evaluated from the original accurately
integrated ray data, not from a pixel approximation of the truth. Thus
$x_h^Td- Tm=(x_h^TGm-Tm)+x_h^T\eta$ is checked against the actual continuum truth.

In [ ]:
ns=np.array([r['n'] for r in rows]); xf=rows[-1]['x']
fig,axs=plt.subplots(2,3,figsize=(13,7),constrained_layout=True)
plots=[('weight norm','weight_norm'),('propagated noise sigma','std'),
       ('pixel delivered-kernel L2 norm','kernel_norm'),('relative L2 target mismatch','relative_mismatch')]
for ax,(label,key) in zip(axs.ravel()[:4],plots):
 ax.loglog(ns,[r[key] for r in rows],'o-'); ax.set_title(label)
axs[1,1].semilogx(ns,[np.linalg.norm(r['x']-xf)/np.linalg.norm(xf) for r in rows],'o-')
axs[1,1].set_title('relative weight change from finest mesh')
axs[1,2].semilogx(ns,[r['target_bias'] for r in rows],'o-',label='delivered truth minus target')
axs[1,2].axhline(0,color='k',lw=.7); axs[1,2].set_title('target error for this one known model')
for ax in axs.ravel(): ax.set_xlabel('pixels per side'); ax.grid(alpha=.2)
plt.show()
fig,ax=plt.subplots(figsize=(8,4))
ax.errorbar(ns,[r['estimate'] for r in rows],yerr=[2*r['std'] for r in rows],fmt='o-',label='estimate +/- 2 propagated sigma')
ax.plot(ns,[r['delivered_truth'] for r in rows],'s--',label='actual delivered truth')
ax.axhline(truth,color='k',label='requested disk mean'); ax.set_xscale('log'); ax.legend(); plt.show()
for r in rows:
 np.testing.assert_allclose(r['estimate']-truth,r['target_bias']+r['noise_draw'],atol=2e-13)
print('Transformed-MDR identity checked on every mesh.')
print('pixel-forward error in the delivered value:',[float(f"{r['pixel_forward_discrepancy']:.3g}") for r in rows])

## Does the normalization matter?

Yes. The preceding experiment imposes unit response. Without it, a different
sequence of estimators is obtained. The following table uses the same physical
target and trade-off but omits the constraint; it should not be confused with
the constrained results.

In [ ]:
print(' n      unconstrained estimate   std        weight norm     constrained estimate')
for r in rows:
 u=r['unconstrained']
 print(f"{r['n']:4d} {u['estimate']:18.6f} {u['std']:10.4g} {u['weight_norm']:12.5g} {r['estimate']:18.6f}")
fig,ax=plt.subplots(figsize=(8,4))
ax.semilogx(ns,[r['estimate'] for r in rows],'o-',label='unit response')
ax.semilogx(ns,[r['unconstrained']['estimate'] for r in rows],'s-',label='no constraint')
ax.axhline(truth,color='k',label='target truth'); ax.legend(); plt.show()

## What this can establish

On a fixed pixel space this is a valid finite-dimensional SOLA problem. The
observed weight and kernel changes show what the chosen question does under
refinement. A rising density-kernel norm is different from divergence of the
finite vector of data weights or from an invalid noise-propagation identity.

Zero-width ray integrals have no bounded extension to L2 of the square. Their
pixel density kernels become concentrated in thin strips. Unit integral alone
does not provide an L2 limit. A finite refinement study illustrates this mechanism
but does not prove an asymptotic limit for all ray geometries.

This is distinct from what happens to a least-squares model. Such models can
converge in L2 to zero while their ray predictions do not tend to zero, precisely
because ray integration is not continuous in L2. Thus "the limit is not in L2"
is not the general explanation. Here we inspect delivered sensitivity densities,
data weights and functional values as three separate objects.

The covariance refers to the delivered property. Neither unit response nor a small
noise standard deviation bounds error in the requested disk mean without further
information about the model. One truth/noise draw is not a coverage experiment.

The companion notebook changes the kernel-matching geometry to H2, on which the
original ray functionals are bounded. It is not simply a more accurate evaluation
of this pixel/L2 criterion.